# MLflow — Experiments, Runs, Registry, Serving

## Mental Model

MLflow is the control plane for the machine learning lifecycle:

- **Tracking** records each experiment run with parameters, metrics, tags, and artifacts.
- **Experiments** organize related runs so you can compare model choices.
- **Artifacts** preserve supporting evidence such as plots, metadata, and serialized models.
- **Model Registry** is the promotion path from candidate model to production model.
- **Serving / Loading** is how downstream engineering systems consume an approved model.

In this notebook, we use MLflow live against **http://localhost:5000** and PostgreSQL telemetry data from a Citi-style monitoring domain:

- 10,000 endpoints
- 500,000 metrics records
- 25,000 alerts
- 6,000+ API endpoints monitored for latency, error rate, and throughput

The goal is to train and track an anomaly detector, compare runs, register the best model, promote it, and load it back for inference.

In [ ]:
import json
import os
import tempfile
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
import psycopg2
from psycopg2.extras import RealDictCursor
from sklearn.ensemble import IsolationForest
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

warnings.filterwarnings("ignore")

TRACKING_URI = "http://localhost:5000"
EXPERIMENT_NAME = "citi-anomaly-detection"
REGISTERED_MODEL_NAME = "citi-anomaly-detector"

DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "dbname": "de_telemetry",
    "user": "de_admin",
    "password": "DeAdmin2026!"
}

mlflow.set_tracking_uri(TRACKING_URI)
print("MLflow tracking URI:", mlflow.get_tracking_uri())

In [ ]:
def get_connection():
    return psycopg2.connect(**DB_CONFIG)

def load_training_frame(limit=20000):
    sql = '''
    WITH metric_base AS (
        SELECT 
            m.endpoint_id,
            MAX(CASE WHEN m.metric_name = 'latency' THEN m.value END) AS latency,
            MAX(CASE WHEN m.metric_name = 'error_rate' THEN m.value END) AS error_rate,
            MAX(CASE WHEN m.metric_name = 'throughput' THEN m.value END) AS throughput,
            MAX(m.timestamp) AS latest_metric_ts
        FROM metrics m
        GROUP BY m.endpoint_id, date_trunc('minute', m.timestamp)
    ),
    alert_flags AS (
        SELECT 
            a.endpoint_id,
            MAX(
                CASE 
                    WHEN a.severity IN ('critical', 'high', 'sev1', 'sev2') THEN 1 
                    ELSE 0 
                END
            ) AS label_alert
        FROM alerts a
        WHERE a.created_at >= NOW() - INTERVAL '30 days'
        GROUP BY a.endpoint_id
    )
    SELECT
        mb.endpoint_id,
        e.region,
        e.status,
        e.category,
        COALESCE(mb.latency, 0.0) AS latency,
        COALESCE(mb.error_rate, 0.0) AS error_rate,
        COALESCE(mb.throughput, 0.0) AS throughput,
        COALESCE(af.label_alert, 0) AS label_alert,
        mb.latest_metric_ts
    FROM metric_base mb
    JOIN endpoints e
      ON e.endpoint_id = mb.endpoint_id
    LEFT JOIN alert_flags af
      ON af.endpoint_id = mb.endpoint_id
    WHERE mb.latency IS NOT NULL
       OR mb.error_rate IS NOT NULL
       OR mb.throughput IS NOT NULL
    ORDER BY mb.latest_metric_ts DESC
    LIMIT %s
    '''
    with get_connection() as conn:
        return pd.read_sql(sql, conn, params=(limit,))

df = load_training_frame(limit=20000)
print("Training frame shape:", df.shape)
display(df.head())

In [ ]:
feature_cols = ["latency", "error_rate", "throughput"]
X = df[feature_cols].fillna(0.0).copy()

# Keep a simple pseudo-ground-truth label from alert severity history so we can compare runs.
# This is not perfect labeling; it's a pragmatic operational proxy.
y_true = df["label_alert"].astype(int)

print("Feature columns:", feature_cols)
print("Positive label proxy count:", int(y_true.sum()))
X.describe()

In [ ]:
def ensure_experiment(name: str) -> str:
    exp = mlflow.get_experiment_by_name(name)
    if exp is None:
        exp_id = mlflow.create_experiment(name)
    else:
        exp_id = exp.experiment_id
    mlflow.set_experiment(name)
    return exp_id

experiment_id = ensure_experiment(EXPERIMENT_NAME)
experiment = mlflow.get_experiment(experiment_id)

print("Experiment ID:", experiment_id)
print("Experiment name:", experiment.name)
print("Experiment artifact location:", experiment.artifact_location)
print("Experiment URL:", f"{TRACKING_URI.rstrip('/')}/#/experiments/{experiment_id}")

In [ ]:
run_summaries = []
best_run = None
best_model = None
best_score = -1.0
best_predictions = None

for contamination in [0.01, 0.05, 0.1]:
    with mlflow.start_run(run_name=f"isolation_forest_cont_{contamination}") as run:
        model = IsolationForest(
            n_estimators=200,
            contamination=contamination,
            random_state=42,
            n_jobs=-1
        )
        model.fit(X)

        raw_pred = model.predict(X)
        # IsolationForest: -1 = anomaly, 1 = normal
        pred_anomaly = np.where(raw_pred == -1, 1, 0)

        cm = confusion_matrix(y_true, pred_anomaly, labels=[0, 1])
        tn, fp, fn, tp = cm.ravel()

        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
        anomaly_rate = float(pred_anomaly.mean())

        mlflow.log_param("algorithm", "IsolationForest")
        mlflow.log_param("contamination", contamination)
        mlflow.log_param("feature_set", ",".join(feature_cols))
        mlflow.log_metric("precision_proxy", precision)
        mlflow.log_metric("recall_proxy", recall)
        mlflow.log_metric("f1_proxy", f1)
        mlflow.log_metric("anomaly_rate", anomaly_rate)
        mlflow.set_tag("domain", "citi-telemetry")
        mlflow.set_tag("tracking_purpose", "anomaly-detection")
        mlflow.set_tag("data_source", "postgresql://localhost:5432/de_telemetry")

        run_info = {
            "run_id": run.info.run_id,
            "contamination": contamination,
            "precision_proxy": precision,
            "recall_proxy": recall,
            "f1_proxy": f1,
            "anomaly_rate": anomaly_rate,
        }
        run_summaries.append(run_info)

        if f1 > best_score:
            best_score = f1
            best_run = run
            best_model = model
            best_predictions = pred_anomaly

run_summary_df = pd.DataFrame(run_summaries).sort_values("f1_proxy", ascending=False).reset_index(drop=True)
display(run_summary_df)
print("Best run ID:", best_run.info.run_id)
print("Best contamination:", float(run_summary_df.iloc[0]['contamination']))

In [ ]:
# Log artifacts for the best run.
# Re-open the best run so all artifacts are attached to the winning run.
best_run_id = best_run.info.run_id

with mlflow.start_run(run_id=best_run_id):
    # Model signature example input
    input_example = X.head(5)

    # Log model
    model_info = mlflow.sklearn.log_model(
        sk_model=best_model,
        artifact_path="model",
        input_example=input_example
    )

    # Approximate feature importance using absolute deviation from learned threshold behavior.
    # IsolationForest doesn't expose classic feature importance, so we use simple dispersion-based proxies
    # to create an operational visualization artifact.
    feature_proxy = X.std().sort_values(ascending=False)

    plt.figure(figsize=(8, 4))
    feature_proxy.plot(kind="bar")
    plt.title("Feature Importance Proxy (Std Dev by Feature)")
    plt.ylabel("Std Dev")
    plt.tight_layout()

    tmp_dir = Path(tempfile.mkdtemp(prefix="mlflow_artifacts_"))
    feature_plot_path = tmp_dir / "feature_importance_proxy.png"
    plt.savefig(feature_plot_path, dpi=140, bbox_inches="tight")
    plt.close()

    mlflow.log_artifact(str(feature_plot_path), artifact_path="plots")

    # Confusion matrix artifact
    cm = confusion_matrix(y_true, best_predictions, labels=[0, 1])
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Normal", "Anomaly"])
    fig, ax = plt.subplots(figsize=(5, 5))
    disp.plot(ax=ax, colorbar=False)
    ax.set_title("Confusion Matrix vs Alert Proxy")
    cm_path = tmp_dir / "confusion_matrix.png"
    fig.savefig(cm_path, dpi=140, bbox_inches="tight")
    plt.close(fig)

    mlflow.log_artifact(str(cm_path), artifact_path="plots")

    # JSON metadata artifact
    metadata = {
        "model_name": REGISTERED_MODEL_NAME,
        "algorithm": "IsolationForest",
        "training_rows": int(len(X)),
        "features": feature_cols,
        "best_run_id": best_run_id,
        "best_f1_proxy": float(best_score),
        "tracking_uri": TRACKING_URI,
        "database": DB_CONFIG["dbname"],
        "citi_narrative": "6000+ API endpoints monitored for latency, error rate, throughput with severity-based alert escalation."
    }
    metadata_path = tmp_dir / "model_metadata.json"
    metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")
    mlflow.log_artifact(str(metadata_path), artifact_path="metadata")

artifact_uri = mlflow.get_run(best_run_id).info.artifact_uri
print("Best run artifact URI:", artifact_uri)
print("Model artifact URI:", model_info.model_uri)

In [ ]:
from mlflow import MlflowClient

client = MlflowClient(tracking_uri=TRACKING_URI)

# Register the model from the winning run.
model_uri_for_registry = f"runs:/{best_run_id}/model"

registered_model = None
try:
    registered_model = client.get_registered_model(REGISTERED_MODEL_NAME)
except Exception:
    registered_model = client.create_registered_model(
        name=REGISTERED_MODEL_NAME,
        description="IsolationForest-based anomaly detector for Citi telemetry metrics."
    )

model_version = client.create_model_version(
    name=REGISTERED_MODEL_NAME,
    source=model_uri_for_registry,
    run_id=best_run_id,
    description="Best tracked run promoted from experiment comparison."
)

version_number = model_version.version

# Add tags and descriptions.
client.set_registered_model_tag(REGISTERED_MODEL_NAME, "domain", "citi-telemetry")
client.set_registered_model_tag(REGISTERED_MODEL_NAME, "owner", "data-engineering")
client.set_model_version_tag(REGISTERED_MODEL_NAME, version_number, "promotion_path", "Staging->Production")
client.update_model_version(
    name=REGISTERED_MODEL_NAME,
    version=version_number,
    description="Production candidate registered from best F1 proxy run."
)

# Transition to Staging then Production.
client.transition_model_version_stage(
    name=REGISTERED_MODEL_NAME,
    version=version_number,
    stage="Staging",
    archive_existing_versions=False
)
client.transition_model_version_stage(
    name=REGISTERED_MODEL_NAME,
    version=version_number,
    stage="Production",
    archive_existing_versions=True
)

versions = client.search_model_versions(f"name = '{REGISTERED_MODEL_NAME}'")
versions_df = pd.DataFrame([
    {
        "name": v.name,
        "version": v.version,
        "stage": v.current_stage,
        "run_id": v.run_id,
        "status": v.status,
        "source": v.source
    }
    for v in versions
]).sort_values("version", ascending=True)

display(versions_df)
print(f"Model {REGISTERED_MODEL_NAME} v{version_number} in Production")

In [ ]:
# Load production model from registry and run inference on 100 fresh rows.
production_model = mlflow.sklearn.load_model(f"models:/{REGISTERED_MODEL_NAME}/Production")

def load_inference_frame(limit=100):
    sql = '''
    SELECT
        e.endpoint_id,
        e.region,
        e.status,
        e.category,
        MAX(CASE WHEN m.metric_name = 'latency' THEN m.value END) AS latency,
        MAX(CASE WHEN m.metric_name = 'error_rate' THEN m.value END) AS error_rate,
        MAX(CASE WHEN m.metric_name = 'throughput' THEN m.value END) AS throughput,
        MAX(m.timestamp) AS latest_metric_ts
    FROM metrics m
    JOIN endpoints e
      ON e.endpoint_id = m.endpoint_id
    GROUP BY e.endpoint_id, e.region, e.status, e.category, date_trunc('minute', m.timestamp)
    ORDER BY latest_metric_ts DESC
    LIMIT %s
    '''
    with get_connection() as conn:
        return pd.read_sql(sql, conn, params=(limit,))

new_df = load_inference_frame(100)
new_X = new_df[feature_cols].fillna(0.0).copy()
new_pred = production_model.predict(new_X)
new_df["is_anomaly"] = np.where(new_pred == -1, 1, 0)

anomaly_count = int(new_df["is_anomaly"].sum())
sample_anomalies = new_df[new_df["is_anomaly"] == 1].head(10)

print("Inference rows:", len(new_df))
print("Anomaly count:", anomaly_count)
display(sample_anomalies)

In [ ]:
# Compare all runs from the experiment using MLflow search APIs.
runs_df = mlflow.search_runs(
    experiment_ids=[experiment_id],
    output_format="pandas"
)

comparison_cols = [
    "run_id",
    "status",
    "tags.mlflow.runName",
    "params.contamination",
    "metrics.precision_proxy",
    "metrics.recall_proxy",
    "metrics.f1_proxy",
    "metrics.anomaly_rate",
    "artifact_uri",
    "start_time",
    "end_time"
]

comparison_df = (
    runs_df[comparison_cols]
    .sort_values("metrics.f1_proxy", ascending=False)
    .reset_index(drop=True)
)

display(comparison_df)

print(
    "Why this matters: search_runs() turns MLflow into a structured experiment table so you can "
    "rank hyperparameter choices, compare operational metrics, and preserve reproducibility."
)

## What Just Happened

MLflow is the experiment tracker that makes ML reproducible.

The model registry is the handoff between **Data Science (training)** and **Data Engineering (serving)**.

In a Citi-style environment, every model that touches production data must be tied to:

- a tracked run,
- a reproducible training context,
- an artifact trail,
- a registered model version,
- and an explicit promotion stage.

That is what turns a notebook experiment into an operational asset.